In [1]:
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr

# 0.Processing

In [5]:
# ========= 路径（你自己改） =========
roi_path  = "/Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis/processed/06.active_map/res/roi_map/working-memory_roi_left_top15_kwx.csv"      # 无表头；第1列ROI名，第2列数值
eeg_path  = "/Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis/res/01.gene_selection/eeg_left_kwx.csv"    # 你截图这种格式：第一列是行名，第一行是列名
out_path  = "/Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis/res/01.gene_selection/eeg_left_top15_kwx.csv"

# ========= 1) 读取 ROI 列表（只取第一列脑区名） =========
roi_df = pd.read_csv(roi_path, header=None)
roi_list = (roi_df.iloc[:, 0].astype(str).str.strip().tolist())

print("ROI list length:", len(roi_list))
print("ROI list:", roi_list)

# ========= 2) 读取 EEG 矩阵 =========
eeg = pd.read_csv(eeg_path, index_col=0)
eeg.index = eeg.index.astype(str).str.strip()
eeg.columns = eeg.columns.astype(str).str.strip()

# 强制数值；空白保持 NaN（不填充）
eeg = eeg.apply(pd.to_numeric, errors="coerce")

print("Original EEG shape:", eeg.shape)

# ========= 3) 找到 EEG 中确实存在的 ROI（保持 roi_list 顺序） =========
roi_in_eeg = [r for r in roi_list if (r in eeg.index and r in eeg.columns)]
print("ROIs found in EEG (kept order):", len(roi_in_eeg))
print("Missing ROIs (not in EEG index/columns):", [r for r in roi_list if r not in roi_in_eeg])

# ========= 4) 截取 10×10 子矩阵并保存 =========
eeg_sub = eeg.loc[roi_in_eeg, roi_in_eeg]
print("Filtered EEG shape:", eeg_sub.shape)

eeg_sub.to_csv(out_path)
print("Saved:", out_path)

ROI list length: 10
ROI list: ['lh.superiorparietal_2', 'lh.rostralmiddlefrontal_1', 'lh.caudalmiddlefrontal_1', 'lh.parsopercularis_1', 'lh.rostralmiddlefrontal_2', 'lh.superiorparietal_3', 'lh.precentral_3', 'lh.superiorfrontal_4', 'lh.precentral_4', 'lh.superiorfrontal_3']
Original EEG shape: (64, 64)
ROIs found in EEG (kept order): 10
Missing ROIs (not in EEG index/columns): []
Filtered EEG shape: (10, 10)
Saved: /Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis/res/01.gene_selection/eeg_left_top15_kwx.csv


In [6]:
# ========= 1) 直接读取你已经算好的 gene co-expression (64×64) =========
# 这个文件应当：第一列是行名(ROI)，第一行是列名(ROI)
gene_coexp_path = "/Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis/res/02.co_expression/gene_co_expr_matrix_top10_kwx.csv"  # <-- 你改
gene_coexp = pd.read_csv(gene_coexp_path, index_col=0)

# （可选）确保数值列为 float；无法转数值的会变成 NaN（不做填充）
gene_coexp = gene_coexp.apply(pd.to_numeric, errors="coerce")

# ========= 2) 读取电刺激矩阵 (64×64) =========
# 按你的截图：第一列是行名(ROI)，列名在第一行，空白会自动读成 NaN
stim_path = "/Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis/res/01.gene_selection/eeg_left_top15_kwx.csv"  # <-- 你改
stim = pd.read_csv(stim_path, index_col=0)
stim = stim.apply(pd.to_numeric, errors="coerce")

# ========= 3) 对齐 ROI（非常重要） =========
# 要求：行列名一致；用交集对齐（保守、不会乱配对）
common = (gene_coexp.index.intersection(gene_coexp.columns).intersection(stim.index).intersection(stim.columns))

if len(common) != 64:
    print(f"[Warning] Common ROI count = {len(common)} (expected 64).")
    # 仍然可以继续算，只是维度可能不是64
    # 如果你希望严格必须64，可以把下面这行改成 raise ValueError(...)
    # raise ValueError(f"ROI不匹配：共同ROI数量={len(common)}，请检查命名/空格/后缀。")

gene_coexp = gene_coexp.loc[common, common]
stim = stim.loc[common, common]

# ========= 4) 上三角展开 + 共同mask（不填充空值） =========
G = gene_coexp.values.astype(float)
S = stim.values.astype(float)

iu = np.triu_indices_from(G, k=1)  # 只取上三角，去对角线
g = G[iu]
s = S[iu]

# 只保留两边都“有值”的边；NaN/inf 全部丢掉
mask = np.isfinite(g) & np.isfinite(s)
g = g[mask]
s = s[mask]

print(f"Used edges for correlation: N={len(g)} (dropped {len(iu[0]) - len(g)} due to NaN/inf)")

[Warning] Common ROI count = 10 (expected 64).
Used edges for correlation: N=43 (dropped 2 due to NaN/inf)


# 1.Correlation

In [7]:
# ========= 5) 计算相关（Pearson + Spearman） =========
r_p, p_p = pearsonr(g, s)
r_s, p_s = spearmanr(g, s)

print(f"Pearson  r={r_p:.4f}, p={p_p:.3e}")
print(f"Spearman r={r_s:.4f}, p={p_s:.3e}")

Pearson  r=0.3549, p=1.954e-02
Spearman r=0.2516, p=1.036e-01


# 2.Save

In [8]:
# ========= 6) （可选）保存对齐后的矩阵，便于复现 =========
out_dir = "/Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis/res/03.correlation/"
gene_coexp.to_csv(out_dir + "gene_coexpr_aligned_kwx.csv")
stim.to_csv(out_dir + "eeg_left_aligned_kwx.csv")

# 3.后9个选择区间的基因共表达和EEG矩阵的相关

In [1]:
import os
import pandas as pd
import numpy as np
from scipy.stats import pearsonr, spearmanr

# =========================================
# 1. 路径设置
# =========================================
base_dir = "/Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis/res"
gene_sel_dir = os.path.join(base_dir, "01.gene_selection")
coexp_dir = os.path.join(base_dir, "02.co_expression")
corr_dir = os.path.join(base_dir, "03.correlation")
os.makedirs(corr_dir, exist_ok=True)

roi_path = "/Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis/processed/06.active_map/res/roi_map/working-memory_roi_left_top15_kwx.csv"
eeg_path = os.path.join(gene_sel_dir, "eeg_left_kwx.csv")
eeg_top15_path = os.path.join(gene_sel_dir, "eeg_left_top15_kwx.csv")

# 你要处理的 9 个区间
percent_ranges = [
    (10, 20),
    (20, 30),
    (30, 40),
    (40, 50),
    (50, 60),
    (60, 70),
    (70, 80),
    (80, 90),
    (90, 100),
]

# =========================================
# 2. 先按你原来的逻辑，生成 EEG 的 top15 子矩阵
#    这一步和你原代码保持一致
# =========================================
roi_df = pd.read_csv(roi_path, header=None)
roi_list = roi_df.iloc[:, 0].astype(str).str.strip().tolist()

print("ROI list length:", len(roi_list))
print("ROI list:", roi_list)

eeg = pd.read_csv(eeg_path, index_col=0)
eeg.index = eeg.index.astype(str).str.strip()
eeg.columns = eeg.columns.astype(str).str.strip()

# 强制数值；空白保持 NaN（不填充）
eeg = eeg.apply(pd.to_numeric, errors="coerce")

print("Original EEG shape:", eeg.shape)

# 保持 roi_list 顺序
roi_in_eeg = [r for r in roi_list if (r in eeg.index and r in eeg.columns)]
print("ROIs found in EEG (kept order):", len(roi_in_eeg))
print("Missing ROIs (not in EEG index/columns):", [r for r in roi_list if r not in roi_in_eeg])

eeg_sub = eeg.loc[roi_in_eeg, roi_in_eeg]
print("Filtered EEG shape:", eeg_sub.shape)

eeg_sub.to_csv(eeg_top15_path)
print("Saved:", eeg_top15_path)

# =========================================
# 3. 循环处理每个基因区间的 gene co-expression 文件
#    下面的相关计算逻辑和你原代码保持一致
# =========================================
results = []

for start_p, end_p in percent_ranges:
    print("\n=====================================")
    print(f"Processing {start_p}-{end_p}% genes")

    # ---------- 输入文件 ----------
    gene_coexp_path = os.path.join(
        coexp_dir,
        f"gene_co_expr_matrix_top{start_p}_{end_p}_kwx.csv"
    )

    stim_path = eeg_top15_path

    print("Gene coexp file:", gene_coexp_path)
    print("EEG file:", stim_path)

    # =========================================
    # 读取 gene co-expression
    # 和你原逻辑一致
    # =========================================
    gene_coexp = pd.read_csv(gene_coexp_path, index_col=0)
    gene_coexp.index = gene_coexp.index.astype(str).str.strip()
    gene_coexp.columns = gene_coexp.columns.astype(str).str.strip()
    gene_coexp = gene_coexp.apply(pd.to_numeric, errors="coerce")

    # =========================================
    # 读取 EEG top15 矩阵
    # 和你原逻辑一致
    # =========================================
    stim = pd.read_csv(stim_path, index_col=0)
    stim.index = stim.index.astype(str).str.strip()
    stim.columns = stim.columns.astype(str).str.strip()
    stim = stim.apply(pd.to_numeric, errors="coerce")

    # =========================================
    # 对齐 ROI（非常重要）
    # 和你原逻辑一致
    # =========================================
    common = (
        gene_coexp.index
        .intersection(gene_coexp.columns)
        .intersection(stim.index)
        .intersection(stim.columns)
    )

    # 你原来写的是 expected 64，但现在你这里实际上是 top15 ROI
    # 所以这里只提示共同 ROI 数量，不强行写死
    print(f"Common ROI count = {len(common)}")

    gene_coexp = gene_coexp.loc[common, common]
    stim = stim.loc[common, common]

    # =========================================
    # 上三角展开 + 共同 mask
    # 和你原逻辑一致
    # =========================================
    G = gene_coexp.values.astype(float)
    S = stim.values.astype(float)

    iu = np.triu_indices_from(G, k=1)  # 只取上三角，去对角线
    g = G[iu]
    s = S[iu]

    mask = np.isfinite(g) & np.isfinite(s)
    g = g[mask]
    s = s[mask]

    print(f"Used edges for correlation: N={len(g)} (dropped {len(iu[0]) - len(g)} due to NaN/inf)")

    # =========================================
    # 计算相关（Pearson + Spearman）
    # 和你原逻辑一致
    # =========================================
    r_p, p_p = pearsonr(g, s)
    r_s, p_s = spearmanr(g, s)

    print(f"Pearson  r={r_p:.4f}, p={p_p:.3e}")
    print(f"Spearman r={r_s:.4f}, p={p_s:.3e}")

    # =========================================
    # 保存每个区间对齐后的矩阵
    # 便于复现
    # =========================================
    gene_aligned_out = os.path.join(
        corr_dir,
        f"gene_coexpr_aligned_top{start_p}_{end_p}_kwx.csv"
    )
    eeg_aligned_out = os.path.join(
        corr_dir,
        f"eeg_left_aligned_top{start_p}_{end_p}_kwx.csv"
    )

    gene_coexp.to_csv(gene_aligned_out)
    stim.to_csv(eeg_aligned_out)

    # =========================================
    # 记录结果
    # =========================================
    results.append({
        "gene_range": f"{start_p}-{end_p}%",
        "n_common_roi": len(common),
        "n_edges_used": len(g),
        "pearson_r": r_p,
        "pearson_p": p_p,
        "spearman_r": r_s,
        "spearman_p": p_s
    })

# =========================================
# 4. 汇总保存所有区间的相关结果
# =========================================
results_df = pd.DataFrame(results)

results_out_path = os.path.join(corr_dir, "gene_eeg_correlation_results_top10_100_kwx.csv")
results_df.to_csv(results_out_path, index=False)

print("\n=====================================")
print("All done.")
print("Summary saved to:")
print(results_out_path)
print(results_df)

ROI list length: 10
ROI list: ['lh.superiorparietal_2', 'lh.rostralmiddlefrontal_1', 'lh.caudalmiddlefrontal_1', 'lh.parsopercularis_1', 'lh.rostralmiddlefrontal_2', 'lh.superiorparietal_3', 'lh.precentral_3', 'lh.superiorfrontal_4', 'lh.precentral_4', 'lh.superiorfrontal_3']
Original EEG shape: (64, 64)
ROIs found in EEG (kept order): 10
Missing ROIs (not in EEG index/columns): []
Filtered EEG shape: (10, 10)
Saved: /Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis/res/01.gene_selection/eeg_left_top15_kwx.csv

Processing 10-20% genes
Gene coexp file: /Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis/res/02.co_expression/gene_co_expr_matrix_top10_20_kwx.csv
EEG file: /Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis/res/01.gene_selection/eeg_left_top15_kwx.csv
Common ROI count = 10
Used edges for correlation: N=43 (dropped 2 due to NaN/inf)
Pearson  r=0.3310, p=3.015e-02
Spearman r=0.2043, p=1.888e-01

Processing 20-30% gen